In [102]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import pickle
import re
from pandas.api.types import CategoricalDtype
# import seaborn as sns

In [4]:
sp_dir = "../_sp_TEST_RESULTS"

In [5]:
os.path.isdir(sp_dir)

True

In [55]:
os.listdir(sp_dir)

['sp_2020-08_KEY2020B-002.pkl',
 'sp_2021-02_KEY2020B-002.pkl',
 'sp_2021-02_KEY2020B-003.pkl',
 'sp_2021-02_KEY2020B-006.pkl',
 'sp_2021-08_KEY2020B-002.pkl',
 'sp_2021-08_KEY2020B-003.pkl',
 'sp_2021-08_KEY2020B-004.pkl',
 'sp_2021-08_KEY2020B-006.pkl',
 'sp_2022-02_KEY2020B-002.pkl',
 'sp_2022-02_KEY2020B-003.pkl',
 'sp_2022-02_KEY2020B-006.pkl',
 'sp_2022-02_KEY2020B-008.pkl']

In [57]:
calib_proposals = [
    "OGG_calib",
    "MuSCAT Commissioning",
    "auto_focus",
    "LCOEngineering",
    "COJ_calib",
    "FLOYDS standards",
    "Photometric standards",
    "standard"
]

In [62]:
def set_duration_category(total_duration):
    if total_duration <= 600:
        return "Short"
    elif total_duration <= 1800:
        return "Medium"
    else:
        return "Long"

duration_type = CategoricalDtype(categories=["Long", "Medium", "Short"], ordered=True)

# df["total_duration_category"] = df["total_duration"].apply(set_duration_category).astype(duration_type)

In [63]:
def set_priority_category(tac_priority):
    if tac_priority <= 15:
        return "Low"
    elif tac_priority <= 29:
        return "Intermediate"
    else:
        return "High"

priority_type = CategoricalDtype(categories=["Low", "Intermediate", "High"], ordered=True)

# df["tac_priority_category"] = df["tac_priority"].apply(set_priority_category).astype(priority_type)

In [72]:
all_dfs = []
all_props = []
all_others = []

for filename in os.listdir(sp_dir):
    filepath = os.path.join(sp_dir, filename)
    yeartext, proposal_id = re.match("sp_(\d{4}-\d{2})_(.+)\.pkl", filename).groups()
    
    baseline_filename = f"baseline_{yeartext}.pkl"
    baseline_filepath = os.path.join("output_files", "baseline", baseline_filename)
    input_filepath = os.path.join("input_files", "baseline", baseline_filename)

    if not os.path.isfile(baseline_filepath):
        print(filepath)
        print("Baseline not present.")
        print()
        continue
    
    print(yeartext, proposal_id)
    print(baseline_filepath)
    print(os.path.isfile(baseline_filepath))
    print(os.path.isfile(input_filepath))
    print()
    
    baseline_data = pickle.load(open(baseline_filepath, "rb"))
    sp_data = pickle.load(open(filepath, "rb"))
    input_data = pickle.load(open(input_filepath, "rb"))

    baseline_requests = set(baseline_data["final_completed_requests"].keys())
    sp_requests = set(sp_data["final_completed_requests"].keys())
    df = input_data["all_requests"]
    proposals = input_data

    df["baseline"] = df["id"].isin(baseline_requests).astype(int)
    df["scheduled"] = df["id"].isin(sp_requests).astype(int)
    df["change"] = df["scheduled"] - df["baseline"]
    df["tac_priority"] = df["proposal_id"].map(input_data["proposals"])
    df["priority_cat"] = df["tac_priority"].apply(set_priority_category).astype(priority_type)
    df["duration_cat"] = df["total_duration"].apply(set_duration_category).astype(duration_type)
    df["filename"] = filename
    df["sim_date"] = yeartext
    df["boosted_proposal"] = proposal_id

    prop_requests = df[df["proposal_id"]==proposal_id]
    other_requests = df[~df["proposal_id"].isin(calib_proposals + [proposal_id])]

    print(f"{proposal_id}:")
    print("Net proposal change:", prop_requests["change"].sum())
    print("Net other change:", other_requests["change"].sum())
    print(prop_requests["change"].value_counts())
    print(other_requests["change"].value_counts())
    print()

    all_dfs.append(df)
    all_props.append(prop_requests)
    all_others.append(other_requests)

2020-08 KEY2020B-002
output_files\baseline\baseline_2020-08.pkl
True
True

KEY2020B-002:
Net proposal change: 42
Net other change: 44
change
 0    972
 1     51
-1      9
Name: count, dtype: int64
change
 0    4609
 1     175
-1     131
Name: count, dtype: int64

2021-02 KEY2020B-002
output_files\baseline\baseline_2021-02.pkl
True
True

KEY2020B-002:
Net proposal change: 205
Net other change: 150
change
 0    1816
 1     279
-1      74
Name: count, dtype: int64
change
 0    9370
 1     821
-1     671
Name: count, dtype: int64

2021-02 KEY2020B-003
output_files\baseline\baseline_2021-02.pkl
True
True

KEY2020B-003:
Net proposal change: 529
Net other change: -100
change
 0    729
 1    538
-1      9
Name: count, dtype: int64
change
 0    10499
-1      678
 1      578
Name: count, dtype: int64

2021-02 KEY2020B-006
output_files\baseline\baseline_2021-02.pkl
True
True

KEY2020B-006:
Net proposal change: 741
Net other change: -138
change
 0    916
 1    754
-1     13
Name: count, dtype: int

In [74]:
fdf = pd.concat(all_dfs)

In [104]:
pdf = pd.concat(all_props)
pdf.head(5)

,id,optimization_type,total_duration,observation_state,location,availability_windows,visibility_windows,config_data,request_group_id,ipp_value,...,win_end,baseline,scheduled,change,tac_priority,priority_cat,duration_cat,filename,sim_date,boosted_proposal
2365543,2365543,TIME,342,COMPLETED,{'telescope_class': '2m0'},"[{'start': '2021-01-24T02:42:00.056128Z', 'end...","{'ogg': datetime.datetime(2021, 1, 24, 10, 50,...","[{'config_type': 'EXPOSE', 'instrument_type': ...",1135948,1.0,...,2021-01-24 16:13:10.449111,1,1,0,35,High,Short,sp_2020-08_KEY2020B-002.pkl,2020-08,KEY2020B-002
2363721,2363721,TIME,342,COMPLETED,{'telescope_class': '2m0'},"[{'start': '2021-01-22T02:32:15.118590Z', 'end...","{'ogg': datetime.datetime(2021, 1, 22, 10, 58,...","[{'config_type': 'EXPOSE', 'instrument_type': ...",1134518,1.0,...,2021-01-22 16:13:20.341424,0,1,1,35,High,Short,sp_2020-08_KEY2020B-002.pkl,2020-08,KEY2020B-002
2361855,2361855,TIME,342,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-20T02:22:15.895254Z', 'end...","{'ogg': datetime.datetime(2021, 1, 20, 11, 5, ...","[{'config_type': 'EXPOSE', 'instrument_type': ...",1133111,1.0,...,2021-01-20 16:13:25.074584,1,1,0,35,High,Short,sp_2020-08_KEY2020B-002.pkl,2020-08,KEY2020B-002
2359777,2359777,TIME,342,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-18T02:12:10.524571Z', 'end...","{'ogg': datetime.datetime(2021, 1, 18, 11, 13,...","[{'config_type': 'EXPOSE', 'instrument_type': ...",1131597,1.0,...,2021-01-18 16:13:24.665614,0,1,1,35,High,Short,sp_2020-08_KEY2020B-002.pkl,2020-08,KEY2020B-002
2358235,2358235,TIME,2445,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-16T08:50:15.048407Z', 'end...","{'coj': datetime.datetime(2021, 1, 16, 15, 47,...","[{'config_type': 'SPECTRUM', 'instrument_type'...",1130262,1.1,...,2021-01-16 18:16:06.898010,1,1,0,35,High,Long,sp_2020-08_KEY2020B-002.pkl,2020-08,KEY2020B-002


In [103]:
odf = pd.concat(all_others)
odf.head(5)

,id,optimization_type,total_duration,observation_state,location,availability_windows,visibility_windows,config_data,request_group_id,ipp_value,...,win_end,baseline,scheduled,change,tac_priority,priority_cat,duration_cat,filename,sim_date,boosted_proposal
2373081,2373081,TIME,3088,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-30T22:03:25Z', 'end': '202...","{'coj': datetime.datetime(2021, 1, 31, 10, 2, ...","[{'config_type': 'LAMP_FLAT', 'instrument_type...",1140990,1.0,...,2021-01-31 14:19:27.806779,0,0,0,16,Intermediate,Long,sp_2020-08_KEY2020B-002.pkl,2020-08,KEY2020B-002
2373019,2373019,TIME,19422,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-31T05:05:00Z', 'end': '202...","{'coj': datetime.datetime(2021, 1, 31, 10, 2, ...","[{'config_type': 'REPEAT_EXPOSE', 'instrument_...",1140950,1.0,...,2021-01-31 10:30:00.000000,1,1,0,30,High,Long,sp_2020-08_KEY2020B-002.pkl,2020-08,KEY2020B-002
2373018,2373018,TIME,3388,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-30T19:02:43Z', 'end': '202...","{'ogg': datetime.datetime(2021, 1, 31, 5, 4, 5...","[{'config_type': 'LAMP_FLAT', 'instrument_type...",1140949,1.0,...,2021-01-31 08:20:29.245336,0,0,0,16,Intermediate,Long,sp_2020-08_KEY2020B-002.pkl,2020-08,KEY2020B-002
2373014,2373014,TIME,3388,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-30T19:01:09Z', 'end': '202...","{'ogg': datetime.datetime(2021, 1, 31, 7, 13, ...","[{'config_type': 'LAMP_FLAT', 'instrument_type...",1140947,1.0,...,2021-01-31 16:11:55.565044,0,0,0,16,Intermediate,Long,sp_2020-08_KEY2020B-002.pkl,2020-08,KEY2020B-002
2372981,2372981,TIME,499,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-30T20:46:00Z', 'end': '202...","{'coj': datetime.datetime(2021, 1, 31, 13, 6, ...","[{'config_type': 'EXPOSE', 'instrument_type': ...",1140914,1.05,...,2021-01-31 13:36:34.306158,0,0,0,20,Intermediate,Short,sp_2020-08_KEY2020B-002.pkl,2020-08,KEY2020B-002


In [79]:
pdf["change"].value_counts()

change
 0    12138
 1     2931
-1      197
Name: count, dtype: int64

In [80]:
odf["change"].value_counts()

change
 0    87518
-1     4845
 1     4719
Name: count, dtype: int64

In [93]:
for filename, g1 in odf.groupby("filename"):
    print(filename)
    for (d_cat, p_cat), g2 in g1.groupby(["duration_cat", "priority_cat"]):
        print("\t", d_cat, p_cat, g2["change"].sum()/len(g2)*100)

sp_2020-08_KEY2020B-002.pkl
	 Long Low -0.684931506849315
	 Long Intermediate 0.4319654427645789
	 Long High -8.0
	 Medium Low 0.0
	 Medium Intermediate 1.7467248908296942
	 Medium High 4.024767801857585
	 Short Low 1.134020618556701
	 Short Intermediate -0.5574136008918618
	 Short High 1.9230769230769231
sp_2021-02_KEY2020B-002.pkl
	 Long Low -0.40816326530612246
	 Long Intermediate 0.8366141732283465
	 Long High -2.4682651622002822
	 Medium Low 0.8163265306122449
	 Medium Intermediate 0.0975609756097561
	 Medium High 3.7719298245614032
	 Short Low 1.5884476534296028
	 Short Intermediate 1.185770750988142
	 Short High 4.996876951905059
sp_2021-02_KEY2020B-003.pkl
	 Long Low 0.40816326530612246
	 Long Intermediate -1.6240157480314958
	 Long High -2.209772798008092
	 Medium Low -1.2244897959183674
	 Medium Intermediate -0.7804878048780488
	 Medium High -0.6386861313868614
	 Short Low 1.7328519855595668
	 Short Intermediate -0.3952569169960474
	 Short High 0.5383580080753702
sp_2021-02_K

In [94]:
for (d_cat, p_cat), group in odf.groupby(["duration_cat", "priority_cat"]):
    print("\t", d_cat, p_cat, group["change"].sum()/len(group)*100)

	 Long Low 0.6691449814126395
	 Long Intermediate -0.9051897545929999
	 Long High -2.9733460762450887
	 Medium Low -0.04553734061930783
	 Medium Intermediate 0.371008347687823
	 Medium High -0.3575259206292456
	 Short Low 1.343680390888841
	 Short Intermediate 1.5228426395939088
	 Short High 2.4341720304271504


In [116]:
boosted = []
others = []

for (priority, duration), g2 in pdf.groupby(["priority_cat", "duration_cat"]):
    subset_count = len(g2)
    diff = g2["change"].sum()
    normed = diff / subset_count
    boosted.append([priority, duration, normed])

for (priority, duration), g2 in odf.groupby(["priority_cat", "duration_cat"]):
    subset_count = len(g2)
    diff = g2["change"].sum()
    normed = diff / subset_count
    others.append([priority, duration, normed])

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes = axes.flatten()

ax=axes[0]
a_s = pd.DataFrame(boosted, columns=["priority", "duration", "change"])
b_s = a_s.pivot(index="priority", columns="duration", values="change").loc[["High"], :]
b_s.rename(columns={"0_Short": "Short", "1_Medium": "Medium", "2_Long": "Long"}, inplace=True)
# sns.heatmap(b_s, annot=True, ax=axes[0])
# ax.set_title("Boosted")
# ax.set_xlabel("Subset Duration")
# ax.set_ylabel("Subset Priority Rank")

# print("All", "Exponential")
ax = axes[1]
a_e = pd.DataFrame(others, columns=["priority", "duration", "change"])
b_e = a_e.pivot(index="priority", columns="duration", values="change").loc[["High", "Intermediate", "Low"], :]
b_e.rename(columns={"0_Short": "Short", "1_Medium": "Medium", "2_Long": "Long"}, inplace=True)

ax.imshow(b_e.values, cmap="viridis", aspect="auto")
# Annotate each cell with the numeric value
for i in range(len(df.index)):
    for j in range(len(df.columns)):
        ax.text(j, i, f"{df.values[i, j]}", ha='center', va='center', color='white')


# sns.heatmap(b_e, annot=True, ax=axes[1])
ax.set_title("Others")
ax.set_xlabel("Subset Duration")
ax.set_ylabel("Subset Priority Rank")


KeyboardInterrupt



Error in callback <function _draw_all_if_interactive at 0x000001455DAC7C70> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 

Error in callback <function flush_figures at 0x00000145959883A0> (for post_execute), with arguments args (),kwargs {}:



KeyboardInterrupt



In [117]:
b_s

duration,Long,Medium,Short
priority,,,
High,0.11374,0.20759,0.295077


In [92]:
for filename, g1 in pdf.groupby("filename"):
    print(filename)
    for (d_cat, p_cat), g2 in g1.groupby(["duration_cat", "priority_cat"]):
        print("\t", d_cat, p_cat, g2["change"].sum()/len(g2)*100)

sp_2020-08_KEY2020B-002.pkl
	 Long High 3.9748953974895396
	 Medium High 3.076923076923077
	 Short High 18.181818181818183
sp_2021-02_KEY2020B-002.pkl
	 Long High 8.080288214101904
	 Medium High 21.73913043478261
	 Short High 15.789473684210526
sp_2021-02_KEY2020B-003.pkl
	 Long High 35.13513513513514
	 Medium High 44.223107569721115
	 Short High 41.73318129988598
sp_2021-02_KEY2020B-006.pkl
	 Long High 42.474489795918366
	 Medium High 50.697674418604656
	 Short High 40.511727078891255
sp_2021-08_KEY2020B-002.pkl
	 Long High 3.1093279839518555
	 Medium High 2.564102564102564
	 Short High 3.3333333333333335
sp_2021-08_KEY2020B-003.pkl
	 Long High 3.508771929824561
	 Medium High 19.828510182207932
	 Short High 16.352201257861633
sp_2021-08_KEY2020B-004.pkl
	 Medium High 17.0
sp_2021-08_KEY2020B-006.pkl
	 Long High 15.384615384615385
	 Medium High 13.161659513590845
	 Short High 22.52252252252252
sp_2022-02_KEY2020B-002.pkl
	 Long High 2.638190954773869
	 Medium High 8.96551724137931
	 Sh